<a href="https://colab.research.google.com/github/fazil-s7/unsupervised_learning/blob/main/clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

Open In Colab

# ============================================================
# MARKET BASKET OPTIMISATION
# ASSOCIATION RULE LEARNING
#
# Algorithms:
# 1. Apriori
# 2. FP-Growth
# 3. ECLAT
#
# Measures:
# Support
# Confidence
# Lift
#
# Includes:
# - Data preprocessing
# - Frequent itemsets
# - Association rules
# - Graphs
# - Algorithm comparison
# ============================================================


# ============================================================
# 1. HIDE JUPYTER DEPRECATION WARNINGS
# ============================================================

import warnings

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning
)

warnings.filterwarnings(
    "ignore",
    message=".*datetime.utcnow.*"
)


# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from itertools import combinations

from mlxtend.preprocessing import TransactionEncoder

from mlxtend.frequent_patterns import (
    apriori,
    fpgrowth,
    association_rules
)

print("Libraries imported successfully!")


# ============================================================
# 3. LOAD DATASET
# ============================================================

file_path = "/content/Market_Basket_Optimisation.csv"

df = pd.read_csv(
    file_path,
    header=None
)

print("\n" + "=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("\nFirst 5 rows:")
print(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nNumber of Transactions:")
print(len(df))


# ============================================================
# 4. CHECK MISSING VALUES
# ============================================================

print("\nMissing Values:")

print(
    df.isnull().sum().sum()
)


# ============================================================
# 5. CONVERT DATA INTO TRANSACTIONS
# ============================================================

transactions = []

for i in range(len(df)):

    items = (
        df.iloc[i]
        .dropna()
        .astype(str)
        .tolist()
    )

    items = [
        item.strip()
        for item in items
        if item.strip() != ""
    ]

    transactions.append(items)


print("\nFirst 5 Transactions:")

for transaction in transactions[:5]:

    print(transaction)


# ============================================================
# 6. TRANSACTION STATISTICS
# ============================================================

total_transactions = len(
    transactions
)

unique_products = set()

total_products = 0

for transaction in transactions:

    total_products += len(
        transaction
    )

    unique_products.update(
        transaction
    )


print("\n" + "=" * 60)
print("TRANSACTION STATISTICS")
print("=" * 60)

print(
    "Total Transactions:",
    total_transactions
)

print(
    "Total Products Purchased:",
    total_products
)

print(
    "Unique Products:",
    len(unique_products)
)


# ============================================================
# 7. TRANSACTION ENCODING
# ============================================================

te = TransactionEncoder()

te_array = te.fit(
    transactions
).transform(
    transactions
)

basket = pd.DataFrame(
    te_array,
    columns=te.columns_
)


print("\nEncoded Dataset:")
print(basket.head())

print("\nEncoded Dataset Shape:")
print(basket.shape)


# ============================================================
# 8. PRODUCT FREQUENCY
# ============================================================

product_frequency = (
    basket.sum()
    .sort_values(
        ascending=False
    )
)


print("\n" + "=" * 60)
print("TOP 20 MOST PURCHASED PRODUCTS")
print("=" * 60)

print(
    product_frequency.head(20)
)


# ============================================================
# 9. GRAPH - TOP PRODUCTS
# ============================================================

plt.figure(
    figsize=(12, 7)
)

product_frequency.head(
    20
).sort_values().plot(
    kind="barh"
)

plt.title(
    "Top 20 Most Purchased Products"
)

plt.xlabel(
    "Number of Transactions"
)

plt.ylabel(
    "Product"
)

plt.tight_layout()

plt.show()


# ============================================================
# 10. APRIORI ALGORITHM
# ============================================================

print("\n" + "=" * 60)
print("APRIORI ALGORITHM")
print("=" * 60)


# Minimum support
MIN_SUPPORT = 0.01


frequent_itemsets_apriori = apriori(
    basket,
    min_support=MIN_SUPPORT,
    use_colnames=True
)


# Add itemset length
frequent_itemsets_apriori[
    "length"
] = frequent_itemsets_apriori[
    "itemsets"
].apply(len)


print(
    "\nNumber of Frequent Itemsets:"
)

print(
    len(frequent_itemsets_apriori)
)


# ============================================================
# 11. APRIORI FREQUENT ITEMSETS
# ============================================================

apriori_sorted = (
    frequent_itemsets_apriori
    .sort_values(
        by="support",
        ascending=False
    )
)


print("\nTop 20 Apriori Frequent Itemsets:")

print(
    apriori_sorted.head(20)
)


# ============================================================
# 12. APRIORI ASSOCIATION RULES
# ============================================================

rules_apriori = association_rules(
    frequent_itemsets_apriori,
    metric="confidence",
    min_threshold=0.20
)


# Sort by lift
rules_apriori = (
    rules_apriori
    .sort_values(
        by="lift",
        ascending=False
    )
)


print(
    "\nNumber of Apriori Rules:"
)

print(
    len(rules_apriori)
)


# ============================================================
# 13. APRIORI TOP RULES
# ============================================================

print("\nTop 20 Apriori Rules:")

print(
    rules_apriori[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift"
        ]
    ].head(20)
)


# ============================================================
# 14. FP-GROWTH
# ============================================================

print("\n" + "=" * 60)
print("FP-GROWTH ALGORITHM")
print("=" * 60)


frequent_itemsets_fp = fpgrowth(
    basket,
    min_support=MIN_SUPPORT,
    use_colnames=True
)


frequent_itemsets_fp[
    "length"
] = frequent_itemsets_fp[
    "itemsets"
].apply(len)


print(
    "\nNumber of FP-Growth Frequent Itemsets:"
)

print(
    len(frequent_itemsets_fp)
)


# ============================================================
# 15. FP-GROWTH ITEMSETS
# ============================================================

fp_sorted = (
    frequent_itemsets_fp
    .sort_values(
        by="support",
        ascending=False
    )
)


print("\nTop 20 FP-Growth Frequent Itemsets:")

print(
    fp_sorted.head(20)
)


# ============================================================
# 16. FP-GROWTH ASSOCIATION RULES
# ============================================================

rules_fp = association_rules(
    frequent_itemsets_fp,
    metric="confidence",
    min_threshold=0.20
)


rules_fp = (
    rules_fp
    .sort_values(
        by="lift",
        ascending=False
    )
)


print(
    "\nNumber of FP-Growth Rules:"
)

print(
    len(rules_fp)
)


print("\nTop 20 FP-Growth Rules:")

print(
    rules_fp[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift"
        ]
    ].head(20)
)


# ============================================================
# 17. ECLAT ALGORITHM
# ============================================================

print("\n" + "=" * 60)
print("ECLAT ALGORITHM")
print("=" * 60)


# ============================================================
# 18. CREATE VERTICAL DATA
# ============================================================

vertical_data = {}

for transaction_id, transaction in enumerate(
    transactions
):

    for item in transaction:

        if item not in vertical_data:

            vertical_data[item] = set()

        vertical_data[item].add(
            transaction_id
        )


print(
    "\nNumber of unique products for ECLAT:"
)

print(
    len(vertical_data)
)


# ============================================================
# 19. ECLAT FUNCTION
# ============================================================

def eclat(
    prefix,
    items,
    min_support_count,
    results
):

    while items:

        item, tidset = items.pop()

        support_count = len(
            tidset
        )

        if support_count >= min_support_count:

            new_itemset = (
                prefix + [item]
            )

            results.append(
                (
                    tuple(new_itemset),
                    support_count
                )
            )

            suffix = []

            for other_item, other_tidset in items:

                intersection = (
                    tidset &
                    other_tidset
                )

                if (
                    len(intersection)
                    >= min_support_count
                ):

                    suffix.append(
                        (
                            other_item,
                            intersection
                        )
                    )

            eclat(
                new_itemset,
                suffix,
                min_support_count,
                results
            )


# ============================================================
# 20. RUN ECLAT
# ============================================================

eclat_results = []

MIN_SUPPORT_COUNT = int(
    MIN_SUPPORT *
    total_transactions
)


items_for_eclat = [
    (item, tids)
    for item, tids
    in vertical_data.items()
]


eclat(
    [],
    items_for_eclat,
    MIN_SUPPORT_COUNT,
    eclat_results
)


# ============================================================
# 21. ECLAT DATAFRAME
# ============================================================

eclat_df = pd.DataFrame(
    eclat_results,
    columns=[
        "itemsets",
        "support_count"
    ]
)


eclat_df["support"] = (
    eclat_df[
        "support_count"
    ] /
    total_transactions
)


eclat_df["length"] = (
    eclat_df[
        "itemsets"
    ].apply(len)
)


eclat_df = (
    eclat_df
    .sort_values(
        by="support",
        ascending=False
    )
)


print(
    "\nNumber of ECLAT Frequent Itemsets:"
)

print(
    len(eclat_df)
)


# ============================================================
# 22. ECLAT FREQUENT ITEMSETS
# ============================================================

print(
    "\nTop 20 ECLAT Frequent Itemsets:"
)

print(
    eclat_df.head(20)
)


# ============================================================
# 23. CREATE ECLAT SUPPORT DICTIONARY
# ============================================================

support_dict = {}

for _, row in eclat_df.iterrows():

    itemset = frozenset(
        row["itemsets"]
    )

    support_dict[itemset] = (
        row["support"]
    )


# ============================================================
# 24. GENERATE ECLAT ASSOCIATION RULES
# ============================================================

eclat_rules = []


for _, row in eclat_df.iterrows():

    itemset = frozenset(
        row["itemsets"]
    )

    support = row["support"]

    if len(itemset) < 2:
        continue

    items = list(itemset)

    for r in range(
        1,
        len(items)
    ):

        for antecedent_tuple in combinations(
            items,
            r
        ):

            antecedent = frozenset(
                antecedent_tuple
            )

            consequent = (
                itemset -
                antecedent
            )

            if len(consequent) == 0:
                continue

            antecedent_support = (
                support_dict.get(
                    antecedent,
                    0
                )
            )

            consequent_support = (
                support_dict.get(
                    consequent,
                    0
                )
            )

            if antecedent_support == 0:
                continue

            if consequent_support == 0:
                continue

            confidence = (
                support /
                antecedent_support
            )

            lift = (
                confidence /
                consequent_support
            )

            eclat_rules.append({

                "antecedents":
                    antecedent,

                "consequents":
                    consequent,

                "support":
                    support,

                "confidence":
                    confidence,

                "lift":
                    lift
            })


# ============================================================
# 25. ECLAT RULE DATAFRAME
# ============================================================

eclat_rules_df = pd.DataFrame(
    eclat_rules
)


if len(eclat_rules_df) > 0:

    eclat_rules_df = (
        eclat_rules_df
        .drop_duplicates()
    )

    eclat_rules_df = (
        eclat_rules_df[
            eclat_rules_df[
                "confidence"
            ] >= 0.20
        ]
    )

    eclat_rules_df = (
        eclat_rules_df
        .sort_values(
            by="lift",
            ascending=False
        )


    )


print(
    "\nNumber of ECLAT Rules:"
)

print(
    len(eclat_rules_df)
)


# ============================================================
# 26. DISPLAY ECLAT RULES
# ============================================================

print(
    "\nTop 20 ECLAT Rules:"
)

if len(eclat_rules_df) > 0:

    print(
        eclat_rules_df[
            [
                "antecedents",
                "consequents",
                "support",
                "confidence",
                "lift"
            ]
        ].head(20)
    )

else:

    print(
        "No ECLAT rules found."
    )


# ============================================================
# 27. FUNCTION TO FORMAT RULES
# ============================================================

def format_rule(row):

    antecedent = ", ".join(
        sorted(
            list(
                row["antecedents"]
            )
        )
    )

    consequent = ", ".join(
        sorted(
            list(
                row["consequents"]
            )
        )
    )

    return (
        antecedent
        + "  --->  "
        + consequent
    )


# ============================================================
# 28. PRINT BEST APRIORI RULES
# ============================================================

print("\n" + "=" * 60)
print("TOP 10 APRIORI RULES")
print("=" * 60)


for _, row in rules_apriori.head(10).iterrows():

    print(
        "\nRule:",
        format_rule(row)
    )

    print(
        "Support:",
        round(
            row["support"],
            4
        )
    )

    print(
        "Confidence:",
        round(
            row["confidence"],
            4
        )
    )

    print(
        "Lift:",
        round(
            row["lift"],
            4
        )
    )


# ============================================================
# 29. PRINT BEST FP-GROWTH RULES
# ============================================================

print("\n" + "=" * 60)
print("TOP 10 FP-GROWTH RULES")
print("=" * 60)


for _, row in rules_fp.head(10).iterrows():

    print(
        "\nRule:",
        format_rule(row)
    )

    print(
        "Support:",
        round(
            row["support"],
            4
        )
    )

    print(
        "Confidence:",
        round(
            row["confidence"],
            4
        )
    )

    print(
        "Lift:",
        round(
            row["lift"],
            4
        )
    )


# ============================================================
# 30. PRINT BEST ECLAT RULES
# ============================================================

print("\n" + "=" * 60)
print("TOP 10 ECLAT RULES")
print("=" * 60)


if len(eclat_rules_df) > 0:

    for _, row in eclat_rules_df.head(10).iterrows():

        print(
            "\nRule:",
            format_rule(row)
        )

        print(
            "Support:",
            round(
                row["support"],
                4
            )
        )

        print(
            "Confidence:",
            round(
                row["confidence"],
                4
            )
        )

        print(
            "Lift:",
            round(
                row["lift"],
                4
            )
        )


# ============================================================
# 31. GRAPH - APRIORI FREQUENT ITEMSETS
# ============================================================

top_apriori = (
    frequent_itemsets_apriori
    .sort_values(
        by="support",
        ascending=False
    )
    .head(15)
    .copy()
)


top_apriori["itemset_name"] = (
    top_apriori[
        "itemsets"
    ].apply(
        lambda x:
        ", ".join(
            sorted(x)
        )
    )
)


plt.figure(
    figsize=(12, 7)
)

sns.barplot(
    data=top_apriori,
    y="itemset_name",
    x="support"
)

plt.title(
    "Top 15 Frequent Itemsets - Apriori"
)

plt.xlabel(
    "Support"
)

plt.ylabel(
    "Itemset"
)

plt.tight_layout()

plt.show()


# ============================================================
# 32. GRAPH - APRIORI CONFIDENCE VS LIFT
# ============================================================

plt.figure(
    figsize=(10, 6)
)

sns.scatterplot(
    data=rules_apriori,
    x="confidence",
    y="lift",
    size="support",
    sizes=(30, 300),
    alpha=0.7
)

plt.title(
    "Apriori - Confidence vs Lift"
)

plt.xlabel(
    "Confidence"
)

plt.ylabel(
    "Lift"
)

plt.grid(
    alpha=0.3
)

plt.show()


# ============================================================
# 33. GRAPH - FP-GROWTH CONFIDENCE VS LIFT
# ============================================================

plt.figure(
    figsize=(10, 6)
)

sns.scatterplot(
    data=rules_fp,
    x="confidence",
    y="lift",
    size="support",
    sizes=(30, 300),
    alpha=0.7
)

plt.title(
    "FP-Growth - Confidence vs Lift"
)

plt.xlabel(
    "Confidence"
)

plt.ylabel(
    "Lift"
)

plt.grid(
    alpha=0.3
)

plt.show()


# ============================================================
# 34. GRAPH - ECLAT CONFIDENCE VS LIFT
# ============================================================

if len(eclat_rules_df) > 0:

    plt.figure(
        figsize=(10, 6)
    )

    sns.scatterplot(
        data=eclat_rules_df,
        x="confidence",
        y="lift",
        size="support",
        sizes=(30, 300),
        alpha=0.7
    )

    plt.title(
        "ECLAT - Confidence vs Lift"
    )

    plt.xlabel(
        "Confidence"
    )

    plt.ylabel(
        "Lift"
    )

    plt.grid(
        alpha=0.3
    )

    plt.show()


# ============================================================
# 35. TOP APRIORI RULES GRAPH
# ============================================================

top_rules = (
    rules_apriori
    .head(10)
    .copy()
)


top_rules["rule"] = (
    top_rules.apply(
        format_rule,
        axis=1
    )
)


plt.figure(
    figsize=(12, 7)
)

sns.barplot(
    data=top_rules,
    y="rule",
    x="lift"
)

plt.title(
    "Top 10 Apriori Rules by Lift"
)

plt.xlabel(
    "Lift"
)

plt.ylabel(
    "Association Rule"
)

plt.tight_layout()

plt.show()


# ============================================================
# 36. ALGORITHM RULE COUNT COMPARISON
# ============================================================

rule_comparison = pd.DataFrame({

    "Algorithm": [
        "Apriori",
        "FP-Growth",
        "ECLAT"
    ],

    "Number_of_Rules": [
        len(rules_apriori),
        len(rules_fp),
        len(eclat_rules_df)
    ]
})


print("\n" + "=" * 60)
print("ALGORITHM COMPARISON")
print("=" * 60)

print(
    rule_comparison
)


# ============================================================
# 37. GRAPH - RULE COUNT COMPARISON
# ============================================================

plt.figure(
    figsize=(8, 5)
)

sns.barplot(
    data=rule_comparison,
    x="Algorithm",
    y="Number_of_Rules"
)

plt.title(
    "Number of Association Rules"
)

plt.xlabel(
    "Algorithm"
)

plt.ylabel(
    "Number of Rules"
)

plt.show()


# ============================================================
# 38. COMPARE BEST LIFT
# ============================================================

best_lift = pd.DataFrame({

    "Algorithm": [
        "Apriori",
        "FP-Growth",
        "ECLAT"
    ],

    "Maximum_Lift": [
        rules_apriori["lift"].max()
        if len(rules_apriori) > 0
        else 0,

        rules_fp["lift"].max()
        if len(rules_fp) > 0
        else 0,

        eclat_rules_df["lift"].max()
        if len(eclat_rules_df) > 0
        else 0
    ]
})


print("\nMaximum Lift Comparison:")

print(
    best_lift
)


# ============================================================
# 39. GRAPH - MAXIMUM LIFT
# ============================================================

plt.figure(
    figsize=(8, 5)
)

sns.barplot(
    data=best_lift,
    x="Algorithm",
    y="Maximum_Lift"
)

plt.title(
    "Maximum Lift Comparison"
)

plt.xlabel(
    "Algorithm"
)

plt.ylabel(
    "Maximum Lift"
)

plt.show()


# ============================================================
# 40. SAVE APRIORI RESULTS
# ============================================================

rules_apriori.to_csv(
    "apriori_association_rules.csv",
    index=False
)

frequent_itemsets_apriori.to_csv(
    "apriori_frequent_itemsets.csv",
    index=False
)


# ============================================================
# 41. SAVE FP-GROWTH RESULTS
# ============================================================

rules_fp.to_csv(
    "fpgrowth_association_rules.csv",
    index=False
)

frequent_itemsets_fp.to_csv(
    "fpgrowth_frequent_itemsets.csv",
    index=False
)


# ============================================================
# 42. SAVE ECLAT RESULTS
# ============================================================

eclat_rules_df.to_csv(
    "eclat_association_rules.csv",
    index=False
)

eclat_df.to_csv(
    "eclat_frequent_itemsets.csv",
    index=False
)


# ============================================================
# 43. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 60)
print("ASSOCIATION RULE LEARNING COMPLETED")
print("=" * 60)

print("\nAlgorithms used:")
print("1. Apriori")
print("2. FP-Growth")
print("3. ECLAT")

print("\nEvaluation measures:")
print("1. Support")
print("2. Confidence")
print("3. Lift")

print("\nFiles created:")
print("1. apriori_association_rules.csv")
print("2. apriori_frequent_itemsets.csv")
print("3. fpgrowth_association_rules.csv")
print("4. fpgrowth_frequent_itemsets.csv")
print("5. eclat_association_rules.csv")
print("6. eclat_frequent_itemsets.csv")

print("\nNumber of Apriori Rules:")
print(len(rules_apriori))

print("\nNumber of FP-Growth Rules:")
print(len(rules_fp))

print("\nNumber of ECLAT Rules:")
print(len(eclat_rules_df))

print("\nAll processing completed successfully!")

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
Libraries imported successfully!

============================================================
DATASET INFORMATION
============================================================

First 5 rows:
              0          1           2                 3             4   \
0         shrimp    almonds     avocado    vegetables mix  green grapes
1        burgers  meatballs        eggs               NaN           NaN
2        chutney        NaN         NaN               NaN           NaN
3         turkey    avocado         NaN               NaN           NaN
4  mineral water       milk  energy bar  whole wheat rice     green tea

                 5     6               7             8             9   \
0  whole weat flour  yams  cottage cheese  energy drink  tomato juice
1               NaN   NaN             NaN           NaN           NaN
2               NaN   NaN             NaN           NaN           NaN
3               NaN   NaN             NaN           NaN           NaN
4               NaN   NaN             NaN           NaN           NaN

               10         11     12     13             14      15  \
0  low fat yogurt  green tea  honey  salad  mineral water  salmon
1             NaN        NaN    NaN    NaN            NaN     NaN
2             NaN        NaN    NaN    NaN            NaN     NaN
3             NaN        NaN    NaN    NaN            NaN     NaN
4             NaN        NaN    NaN    NaN            NaN     NaN

                  16               17       18         19
0  antioxydant juice  frozen smoothie  spinach  olive oil
1                NaN              NaN      NaN        NaN
2                NaN              NaN      NaN        NaN
3                NaN              NaN      NaN        NaN
4                NaN              NaN      NaN        NaN

Dataset Shape:
(7501, 20)

Number of Transactions:
7501

Missing Values:
120657

First 5 Transactions:
['shrimp', 'almonds', 'avocado', 'vegetables mix', 'green grapes', 'whole weat flour', 'yams', 'cottage cheese', 'energy drink', 'tomato juice', 'low fat yogurt', 'green tea', 'honey', 'salad', 'mineral water', 'salmon', 'antioxydant juice', 'frozen smoothie', 'spinach', 'olive oil']
['burgers', 'meatballs', 'eggs']
['chutney']
['turkey', 'avocado']
['mineral water', 'milk', 'energy bar', 'whole wheat rice', 'green tea']

============================================================
TRANSACTION STATISTICS
============================================================
Total Transactions: 7501
Total Products Purchased: 29363
Unique Products: 119

Encoded Dataset:
   almonds  antioxydant juice  asparagus  avocado  babies food  bacon  \
0     True               True      False     True        False  False
1    False              False      False    False        False  False
2    False              False      False    False        False  False
3    False              False      False     True        False  False
4    False              False      False    False        False  False

   barbecue sauce  black tea  blueberries  body spray  ...  turkey  \
0           False      False        False       False  ...   False
1           False      False        False       False  ...   False
2           False      False        False       False  ...   False
3           False      False        False       False  ...    True
4           False      False        False       False  ...   False

   vegetables mix  water spray  white wine  whole weat flour  \
0            True        False       False              True
1           False        False       False             False
2           False        False       False             False
3           False        False       False             False
4           False        False       False             False

   whole wheat pasta  whole wheat rice   yams  yogurt cake  zucchini
0              False             False   True        False     False
1              False             False  False        False     False
2              False             False  False        False     False
3              False             False  False        False     False
4              False              True  False        False     False

[5 rows x 119 columns]

Encoded Dataset Shape:
(7501, 119)

============================================================
TOP 20 MOST PURCHASED PRODUCTS
============================================================
mineral water        1788
eggs                 1348
spaghetti            1306
french fries         1282
chocolate            1229
green tea             991
milk                  972
ground beef           737
frozen vegetables     715
pancakes              713
burgers               654
cake                  608
cookies               603
escalope              595
low fat yogurt        574
shrimp                536
tomatoes              513
olive oil             494
frozen smoothie       475
turkey                469
dtype: int64

============================================================
APRIORI ALGORITHM
============================================================

Number of Frequent Itemsets:
257

Top 20 Apriori Frequent Itemsets:
     support             itemsets  length
46  0.238368      (mineral water)       1
19  0.179709               (eggs)       1
63  0.174110          (spaghetti)       1
24  0.170911       (french fries)       1
13  0.163845          (chocolate)       1
32  0.132116          (green tea)       1
45  0.129583               (milk)       1
33  0.098254        (ground beef)       1
30  0.095321  (frozen vegetables)       1
53  0.095054           (pancakes)       1
6   0.087188            (burgers)       1
8   0.081056               (cake)       1
15  0.080389            (cookies)       1
22  0.079323           (escalope)       1
41  0.076523     (low fat yogurt)       1
61  0.071457             (shrimp)       1
67  0.068391           (tomatoes)       1
52  0.065858          (olive oil)       1
29  0.063325    (frozen smoothie)       1
68  0.062525             (turkey)       1

Number of Apriori Rules:
162

Top 20 Apriori Rules:
                            antecedents          consequents   support  \
75                      (herb & pepper)        (ground beef)  0.015998
154          (spaghetti, mineral water)        (ground beef)  0.017064
69                           (tomatoes)  (frozen vegetables)  0.016131
67                             (shrimp)  (frozen vegetables)  0.016664
143               (milk, mineral water)  (frozen vegetables)  0.011065
153        (ground beef, mineral water)          (spaghetti)  0.017064
145  (mineral water, frozen vegetables)               (milk)  0.011065
149               (milk, mineral water)        (ground beef)  0.011065
90                               (soup)               (milk)  0.015198
80                          (spaghetti)        (ground beef)  0.039195
79                        (ground beef)          (spaghetti)  0.039195
70                      (grated cheese)        (ground beef)  0.011332
78                          (olive oil)        (ground beef)  0.014131
129              (spaghetti, chocolate)               (milk)  0.010932
159          (olive oil, mineral water)          (spaghetti)  0.010265
133                 (ground beef, eggs)      (mineral water)  0.010132
123          (mineral water, chocolate)        (ground beef)  0.010932
150                 (milk, ground beef)      (mineral water)  0.011065
146          (spaghetti, mineral water)  (frozen vegetables)  0.011998
108                          (red wine)          (spaghetti)  0.010265

     confidence      lift
75     0.323450  3.291994
154    0.285714  2.907928
69     0.235867  2.474464
67     0.233209  2.446574
143    0.230556  2.418737
153    0.416938  2.394681
145    0.309701  2.389991
149    0.230556  2.346536
90     0.300792  2.321232
80     0.225115  2.291162
79     0.398915  2.291162
70     0.216285  2.201294
78     0.214575  2.183889
129    0.278912  2.152382
159    0.371981  2.136468
133    0.506667  2.125563
123    0.207595  2.112849
150    0.503030  2.110308
146    0.200893  2.107549
108    0.364929  2.095966

============================================================
FP-GROWTH ALGORITHM
============================================================

Number of FP-Growth Frequent Itemsets:
257

Top 20 FP-Growth Frequent Itemsets:
     support             itemsets  length
0   0.238368      (mineral water)       1
15  0.179709               (eggs)       1
26  0.174110          (spaghetti)       1
22  0.170911       (french fries)       1
31  0.163845          (chocolate)       1
1   0.132116          (green tea)       1
19  0.129583               (milk)       1
50  0.098254        (ground beef)       1
27  0.095321  (frozen vegetables)       1
43  0.095054           (pancakes)       1
16  0.087188            (burgers)       1
56  0.081056               (cake)       1
28  0.080389            (cookies)       1
51  0.079323           (escalope)       1
2   0.076523     (low fat yogurt)       1
3   0.071457             (shrimp)       1
34  0.068391           (tomatoes)       1
4   0.065858          (olive oil)       1
5   0.063325    (frozen smoothie)       1
18  0.062525             (turkey)       1

Number of FP-Growth Rules:
162

Top 20 FP-Growth Rules:
                            antecedents          consequents   support  \
154                     (herb & pepper)        (ground beef)  0.015998
141          (spaghetti, mineral water)        (ground beef)  0.017064
116                          (tomatoes)  (frozen vegetables)  0.016131
9                              (shrimp)  (frozen vegetables)  0.016664
84                (milk, mineral water)  (frozen vegetables)  0.011065
140        (ground beef, mineral water)          (spaghetti)  0.017064
86   (mineral water, frozen vegetables)               (milk)  0.011065
142               (milk, mineral water)        (ground beef)  0.011065
65                               (soup)               (milk)  0.015198
135                         (spaghetti)        (ground beef)  0.039195
134                       (ground beef)          (spaghetti)  0.039195
130                     (grated cheese)        (ground beef)  0.011332
14                          (olive oil)        (ground beef)  0.014131
53               (spaghetti, chocolate)               (milk)  0.010932
16           (olive oil, mineral water)          (spaghetti)  0.010265
148                 (ground beef, eggs)      (mineral water)  0.010132
147          (mineral water, chocolate)        (ground beef)  0.010932
143                 (milk, ground beef)      (mineral water)  0.011065
81           (spaghetti, mineral water)  (frozen vegetables)  0.011998
120                          (red wine)          (spaghetti)  0.010265

     confidence      lift
154    0.323450  3.291994
141    0.285714  2.907928
116    0.235867  2.474464
9      0.233209  2.446574
84     0.230556  2.418737
140    0.416938  2.394681
86     0.309701  2.389991
142    0.230556  2.346536
65     0.300792  2.321232
135    0.225115  2.291162
134    0.398915  2.291162
130    0.216285  2.201294
14     0.214575  2.183889
53     0.278912  2.152382
16     0.371981  2.136468
148    0.506667  2.125563
147    0.207595  2.112849
143    0.503030  2.110308
81     0.200893  2.107549
120    0.364929  2.095966

============================================================
ECLAT ALGORITHM
============================================================

Number of unique products for ECLAT:
119

Number of ECLAT Frequent Itemsets:
260

Top 20 ECLAT Frequent Itemsets:
                 itemsets  support_count   support  length
242      (mineral water,)           1788  0.238368       1
222               (eggs,)           1348  0.179709       1
151          (spaghetti,)           1306  0.174110       1
189       (french fries,)           1282  0.170911       1
116          (chocolate,)           1229  0.163845       1
249          (green tea,)            991  0.132116       1
206               (milk,)            972  0.129583       1
44         (ground beef,)            737  0.098254       1
173  (frozen vegetables,)            715  0.095321       1
72            (pancakes,)            713  0.095054       1
231            (burgers,)            654  0.087188       1
19                (cake,)            608  0.081056       1
147            (cookies,)            603  0.080389       1
38            (escalope,)            595  0.079323       1
251     (low fat yogurt,)            574  0.076523       1
259             (shrimp,)            536  0.071457       1
96            (tomatoes,)            513  0.068391       1
234          (olive oil,)            494  0.065858       1
237    (frozen smoothie,)            475  0.063325       1
217             (turkey,)            469  0.062525       1

Number of ECLAT Rules:
162

Top 20 ECLAT Rules:
                            antecedents          consequents   support  \
170                     (herb & pepper)        (ground beef)  0.015998
147          (spaghetti, mineral water)        (ground beef)  0.017064
169                          (tomatoes)  (frozen vegetables)  0.016131
152                            (shrimp)  (frozen vegetables)  0.016664
353               (milk, mineral water)  (frozen vegetables)  0.011065
146        (ground beef, mineral water)          (spaghetti)  0.017064
355  (mineral water, frozen vegetables)               (milk)  0.011065
347               (milk, mineral water)        (ground beef)  0.011065
191                              (soup)               (milk)  0.015198
11                          (spaghetti)        (ground beef)  0.039195
10                        (ground beef)          (spaghetti)  0.039195
333                     (grated cheese)        (ground beef)  0.011332
223                         (olive oil)        (ground beef)  0.014131
369              (spaghetti, chocolate)               (milk)  0.010932
408          (olive oil, mineral water)          (spaghetti)  0.010265
419                 (ground beef, eggs)      (mineral water)  0.010132
363          (mineral water, chocolate)        (ground beef)  0.010932
346                 (ground beef, milk)      (mineral water)  0.011065
291          (spaghetti, mineral water)  (frozen vegetables)  0.011998
413                          (red wine)          (spaghetti)  0.010265

     confidence      lift
170    0.323450  3.291994
147    0.285714  2.907928
169    0.235867  2.474464
152    0.233209  2.446574
353    0.230556  2.418737
146    0.416938  2.394681
355    0.309701  2.389991
347    0.230556  2.346536
191    0.300792  2.321232
11     0.225115  2.291162
10     0.398915  2.291162
333    0.216285  2.201294
223    0.214575  2.183889
369    0.278912  2.152382
408    0.371981  2.136468
419    0.506667  2.125563
363    0.207595  2.112849
346    0.503030  2.110308
291    0.200893  2.107549
413    0.364929  2.095966

============================================================
TOP 10 APRIORI RULES
============================================================

Rule: herb & pepper  --->  ground beef
Support: 0.016
Confidence: 0.3235
Lift: 3.292

Rule: mineral water, spaghetti  --->  ground beef
Support: 0.0171
Confidence: 0.2857
Lift: 2.9079

Rule: tomatoes  --->  frozen vegetables
Support: 0.0161
Confidence: 0.2359
Lift: 2.4745

Rule: shrimp  --->  frozen vegetables
Support: 0.0167
Confidence: 0.2332
Lift: 2.4466

Rule: milk, mineral water  --->  frozen vegetables
Support: 0.0111
Confidence: 0.2306
Lift: 2.4187

Rule: ground beef, mineral water  --->  spaghetti
Support: 0.0171
Confidence: 0.4169
Lift: 2.3947

Rule: frozen vegetables, mineral water  --->  milk
Support: 0.0111
Confidence: 0.3097
Lift: 2.39

Rule: milk, mineral water  --->  ground beef
Support: 0.0111
Confidence: 0.2306
Lift: 2.3465

Rule: soup  --->  milk
Support: 0.0152
Confidence: 0.3008
Lift: 2.3212

Rule: spaghetti  --->  ground beef
Support: 0.0392
Confidence: 0.2251
Lift: 2.2912

============================================================
TOP 10 FP-GROWTH RULES
============================================================

Rule: herb & pepper  --->  ground beef
Support: 0.016
Confidence: 0.3235
Lift: 3.292

Rule: mineral water, spaghetti  --->  ground beef
Support: 0.0171
Confidence: 0.2857
Lift: 2.9079

Rule: tomatoes  --->  frozen vegetables
Support: 0.0161
Confidence: 0.2359
Lift: 2.4745

Rule: shrimp  --->  frozen vegetables
Support: 0.0167
Confidence: 0.2332
Lift: 2.4466

Rule: milk, mineral water  --->  frozen vegetables
Support: 0.0111
Confidence: 0.2306
Lift: 2.4187

Rule: ground beef, mineral water  --->  spaghetti
Support: 0.0171
Confidence: 0.4169
Lift: 2.3947

Rule: frozen vegetables, mineral water  --->  milk
Support: 0.0111
Confidence: 0.3097
Lift: 2.39

Rule: milk, mineral water  --->  ground beef
Support: 0.0111
Confidence: 0.2306
Lift: 2.3465

Rule: soup  --->  milk
Support: 0.0152
Confidence: 0.3008
Lift: 2.3212

Rule: spaghetti  --->  ground beef
Support: 0.0392
Confidence: 0.2251
Lift: 2.2912

============================================================
TOP 10 ECLAT RULES
============================================================

Rule: herb & pepper  --->  ground beef
Support: 0.016
Confidence: 0.3235
Lift: 3.292

Rule: mineral water, spaghetti  --->  ground beef
Support: 0.0171
Confidence: 0.2857
Lift: 2.9079

Rule: tomatoes  --->  frozen vegetables
Support: 0.0161
Confidence: 0.2359
Lift: 2.4745

Rule: shrimp  --->  frozen vegetables
Support: 0.0167
Confidence: 0.2332
Lift: 2.4466

Rule: milk, mineral water  --->  frozen vegetables
Support: 0.0111
Confidence: 0.2306
Lift: 2.4187

Rule: ground beef, mineral water  --->  spaghetti
Support: 0.0171
Confidence: 0.4169
Lift: 2.3947

Rule: frozen vegetables, mineral water  --->  milk
Support: 0.0111
Confidence: 0.3097
Lift: 2.39

Rule: milk, mineral water  --->  ground beef
Support: 0.0111
Confidence: 0.2306
Lift: 2.3465

Rule: soup  --->  milk
Support: 0.0152
Confidence: 0.3008
Lift: 2.3212

Rule: spaghetti  --->  ground beef
Support: 0.0392
Confidence: 0.2251
Lift: 2.2912





============================================================
ALGORITHM COMPARISON
============================================================
   Algorithm  Number_of_Rules
0    Apriori              162
1  FP-Growth              162
2      ECLAT              162

Maximum Lift Comparison:
   Algorithm  Maximum_Lift
0    Apriori      3.291994
1  FP-Growth      3.291994
2      ECLAT      3.291994


============================================================
ASSOCIATION RULE LEARNING COMPLETED
============================================================

Algorithms used:
1. Apriori
2. FP-Growth
3. ECLAT

Evaluation measures:
1. Support
2. Confidence
3. Lift

Files created:
1. apriori_association_rules.csv
2. apriori_frequent_itemsets.csv
3. fpgrowth_association_rules.csv
4. fpgrowth_frequent_itemsets.csv
5. eclat_association_rules.csv
6. eclat_frequent_itemsets.csv

Number of Apriori Rules:
162

Number of FP-Growth Rules:
162

Number of ECLAT Rules:
162

All processing completed successfully!